In [1]:
import polars as pl
from rapidfuzz import fuzz

In [2]:
answers = pl.read_parquet("../data_sampled/cases_answer_key.parquet")
corrupted = pl.read_parquet("../data_sampled/corrupted_cases_labeled.parquet")
clean = pl.read_parquet("../data_sampled/cases_sampled.parquet")

In [3]:
clean.lazy().collect_schema()

Schema([('ddl_case_id', String),
        ('year', Int64),
        ('state_code', Int64),
        ('dist_code', Int64),
        ('court_no', Int64),
        ('cino', String),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('type_name', Int64),
        ('purpose_name', Int64),
        ('disp_name', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('date_first_list', Date),
        ('date_last_list', Date),
        ('date_next_list', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('ddl_filing_judge_id', Int64),
        ('ddl_decision_judge_id', Int64),
        ('female_judge_filing', String),
        ('start_date_filing', Date),
        ('end_da

In [4]:
corrupted.lazy().collect_schema()

Schema([('record_id', UInt32),
        ('year', Int64),
        ('judge_position', String),
        ('female_defendant', String),
        ('female_petitioner', String),
        ('female_adv_def', Int64),
        ('female_adv_pet', Int64),
        ('date_of_filing', Date),
        ('date_of_decision', Date),
        ('type_name_s', String),
        ('disp_name_s', String),
        ('purpose_name_s', String),
        ('state_name', String),
        ('district_name', String),
        ('court_name', String),
        ('female_judge_filing', String),
        ('female_judge_decision', String),
        ('act_s', List(String)),
        ('section_s', List(String)),
        ('number_sections_ipc', Int64),
        ('criminal', Int64),
        ('bailable_ipc', String),
        ('n_acts_or_sections', UInt32)])

In [5]:
clean = clean.select(["ddl_case_id",
        'year',
        'judge_position',
        'female_defendant',
        'female_petitioner',
        'female_adv_def',
        'female_adv_pet',
        'date_of_filing',
        'date_of_decision', 
        'type_name_s', 
        'disp_name_s', 
        'purpose_name_s', 
        'state_name', 
        'district_name', 
        'court_name', 
        'female_judge_filing', 
        'female_judge_decision', 
        'act_s', 
        'section_s', 
        'number_sections_ipc', 
        'criminal', 
        'bailable_ipc', 
        'n_acts_or_sections'])

In [6]:
print(corrupted.shape)
print(clean.shape)
print(answers.shape)

(50000, 23)
(50000, 23)
(50000, 2)


In [7]:
join = []
for c in corrupted.columns:
    if c not in ["record_id", "act_s", "section_s"]:
        join.append(c)

In [8]:
matches = corrupted.join(clean, on=join, how="inner")
print(matches.height)

4829


In [9]:
matches = corrupted.join(clean, on=join, how="inner", nulls_equal=True)
print(matches.height)

27887


In [10]:
matches.select(pl.col("record_id").n_unique())

record_id
u32
27270


In [11]:
matches_joined = matches.join(answers, on="record_id", how="left", suffix="_true")
pairs = matches_joined.filter(pl.col("ddl_case_id") == pl.col("ddl_case_id_true"))
print(pairs.height)

27255


In [12]:
precision = pairs.height / 27_887
print(f"precision percent of {precision * 100} percent via exact column-column matching.")
correctly_found = pairs.height / 50_000
print(f"recall percent of {correctly_found * 100} percent of original via exactly column-column matching.")

precision percent of 97.73371104815864 percent via exact column-column matching.
recall percent of 54.510000000000005 percent of original via exactly column-column matching.


In [13]:
clean.group_by("state_name").agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings")).select(pl.col("pairings").sum())

pairings
u32
204713292


In [14]:
per_state = clean.group_by("state_name").agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings"))

per_state.sort("pairings", descending=True)

state_name,len,pairings
str,u32,u32
"""Uttar Pradesh""",9847,96963409
"""Bihar""",4532,20539024
"""Maharashtra""",4250,18062500
"""Madhya Pradesh""",3390,11492100
"""Rajasthan""",3383,11444689
…,…,…
"""Sikkim""",20,400
"""Meghalaya""",7,49
"""DNH at Silvasa""",6,36


In [15]:
print(50_000 * 50_000)
print((50_000 * 50_000) / 204713292)

2500000000
12.212201638572644


In [16]:
per_state_and_judgep = clean.group_by(["state_name", "judge_position"]).agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings"))

In [17]:
per_state_and_judgep.sort("pairings", descending = True)

state_name,judge_position,len,pairings
str,str,u32,u32
"""Uttar Pradesh""","""chief judicial magistrate""",4774,22791076
"""Uttar Pradesh""","""district and sessions court""",2255,5085025
"""Bihar""","""judicial magistrate court""",2177,4739329
"""Madhya Pradesh""","""district and sessions court""",1947,3790809
"""Bihar""","""district and sessions court""",1729,2989441
…,…,…,…
"""Kerala""","""judicial first class magistrat…",1,1
"""Assam""","""glt_cj_estt""",1,1
"""Rajasthan""","""acd court""",1,1


In [18]:
per_state_and_judgep.select(pl.col("pairings").sum())

pairings
u32
60203136


In [19]:
per_state_and_judge_and_outcome = clean.group_by(["state_name", "judge_position", "disp_name_s"]).agg(pl.len()).with_columns((pl.col("len") * pl.col("len")).alias("pairings"))

In [20]:
per_state_and_judge_and_outcome.select(pl.col("pairings").sum())

pairings
u32
19629930


In [21]:
clean_trimmed = clean.select(["ddl_case_id",
                              "state_name",
                              "judge_position",
                              "disp_name_s",
                              "date_of_filing",
                              "date_of_decision",
                              "district_name",
                              "court_name",
                              "type_name_s",
                              "female_defendant",
                              "female_petitioner"
                                ])

corrupted_trimmed = corrupted.select(["record_id",
                              "state_name",
                              "judge_position",
                              "disp_name_s",
                              "date_of_filing",
                              "date_of_decision",
                              "district_name",
                              "court_name",
                              "type_name_s",
                              "female_defendant",
                              "female_petitioner"
                                ])

In [22]:
both_joined = corrupted_trimmed.join(clean_trimmed, on=["state_name", "judge_position", "disp_name_s"], how="inner", suffix="_clean")
both_joined.height

19629930

In [23]:
check_matches = both_joined.join(answers, on="record_id", how="left", suffix="_true")
matches = check_matches.filter(pl.col("ddl_case_id") == pl.col("ddl_case_id_true"))
print(matches.height)

50000


In [24]:
filing_diff = both_joined.with_columns((pl.col("date_of_filing") - pl.col("date_of_filing_clean")).dt.total_days().abs().alias("filing_diff"))
decision_diff = both_joined.with_columns((pl.col("date_of_decision") - pl.col("date_of_decision_clean")).dt.total_days().abs().alias("decision_diff"))

